# Cleaning
not done yet

# Setup & Imports

In [ ]:
print("Loading Libraries")
# Core Libraries
import pandas as pd # excel tools
import numpy as np # math
print("Done loading")


# Load Data from Excel Sheet
Loading from the Sheet and organising it through their sheets

In [ ]:
#reading from excel files
print('Loading sheets')
print('clinical')
# clinical data
clincal_sheet = pd.read_excel('../RA/RA_MAP_Clinical_Figshare_17_5_21.xlsx', sheet_name=None)
print('protogen')
# protein analysis
protogen_sheet = pd.read_excel('../RA/Protogen_RA_MAP_16_05_21.xlsx', sheet_name=None)
print('somascan')
# ra-responder and non responder
somascan_sheet = pd.read_excel('../RA/13322471/SOMASCAN_RA-Map_figshare_17_11_20.xlsx', sheet_name=None)
# all_sheets at once just in case its useful
print('Indexing sheets')
all_sheets = {
    'clinical': clincal_sheet,
    'protogen': protogen_sheet,
    'somascan': somascan_sheet
}
print('Done')

# Structure
Take a look at the structure and initialize variables for them

In [ ]:
#print all sheetnames
print(clincal_sheet.keys())
print(protogen_sheet.keys())
print(somascan_sheet.keys())


In [ ]:
# indexing again
# clinical
df_clinical = clincal_sheet['OpenPseudonymised_RA_MAP_Clinic']
df_steroids = clincal_sheet['intramuscular steroids']
df_meds = clincal_sheet['RA Meds']
df_glossary = clincal_sheet['Glossary'] 
# protogen/proteins
df_Samples = protogen_sheet['LIS_PG665-P01 RA MAP Samples Ex']
df_Samples_Annotation = protogen_sheet['Sample annotation']
# somascan
df_expMatrix = somascan_sheet['expression matrix']
df_sampMatrix = somascan_sheet['sample matrix']

# important dfs that we will work with
df_imp = {
    'df_clinical': df_clinical,
    'df_steroids': df_steroids,
    'df_meds': df_meds,
    'df_expMatrix': df_expMatrix,
    'df_sampMatrix': df_sampMatrix    
}

#show names
for source_name, sheets in all_sheets.items():
    print(f'=== {source_name} ===')
    for sheet_name, df in sheets.items():
        print(f'  {sheet_name}: {df.shape[0]} rows x {df.shape[1]} columns')
    print()

In [ ]:
# first look and dtypes
print(df_clinical.head())
display(df_clinical['PAIN.0M'].head())
with pd.option_context('display.max_rows', None):
    print(df_clinical.dtypes)

# Cleanup
starting from top to bottom

In [ ]:
def data_quality_report(df):
    """Function to make a Dataqualityreport with df as Input"""
    
    print("Dataquality report")
    print()
    print(f"Rows: {len(df)} | Columns: {len(df.columns)}")
    print(f"Total of {len(df) * len(df.columns)} data")

    
    print("")
    # 1. Missing values per Row
    print("Missing Values")
    missing = df.isna().sum() # total sum of missing values
    missing_pct = (missing / len(df) * 100).round(2) 
    missing_report = pd.DataFrame({ # new dataframe for missing values
        'Missing': missing,
        'Percentage': missing_pct
    })
    # Only show rows with missing values
    problems = missing_report[missing_report['Missing'] > 0]
    if len(problems) > 0:
        print(problems.sort_values('Missing', ascending=False))
    else:
        print("No missing values were found!")
    print()
    
    # 2. Duplikate
    print("How many duplicates?")
    print(f"Duplicates: {df.duplicated().sum()}")
    print()
    
    # 3. Datentypen-Überblick
    print("Datatypes")
    for dtype in df.dtypes.unique():
        rows = df.select_dtypes(include=[dtype, 'str']).columns.tolist()
        print(f"{dtype}: {len(rows)} Rows")
        print(f"{rows[:5]}{'...' if len(rows) > 5 else ''}")
    print()
    
    # 4. Numerical Values special cases and distribution
    print("Numerical Values")
    num_cols = df.select_dtypes(include=[np.number]).columns
    for col in num_cols:
        q1 = df[col].quantile(0.25)
        q3 = df[col].quantile(0.75)
        iqr = q3 - q1
        special = ((df[col] < q1 - 1.5 * iqr) | (df[col] > q3 + 1.5 * iqr)).sum()
        print(f"{col}:")
        print(f"  Min: {df[col].min()} | Max: {df[col].max()} | "
              f"Mean: {df[col].mean():.2f} | Special values: {special}")
    print()
    
    # 5. String-Spalten: Konsistenz
    print("Strings")
    str_cols = df.select_dtypes(include=['object', 'str']).columns
    for col in str_cols:
        unique = df[col].nunique()
        print(f"{col}:")
        print(f"Unique Strings: {unique}")
        
        # Leerzeichen-Probleme
        if df[col].dropna().str.startswith(' ').any() or df[col].dropna().str.endswith(' ').any():
            print(f"Enthält führende/nachfolgende Leerzeichen!")
        
        # Groß-/Kleinschreibung inkonsistent?
        if unique != df[col].str.lower().nunique():
            print(f"Inkonsistente Groß-/Kleinschreibung!")
        
        # Bei wenigen einzigartigen Werten: zeig sie
        if unique <= 15:
            print(f"  Werte: {df[col].value_counts().to_dict()}")
    print()

In [ ]:
import sys
sys.stdout = sys.__stderr__
print("kann ich das lesen?")

In [ ]:
print(f'Patientinnen mit Steroiden: {df_steroids["Digest"].nunique()}')
print(f'Gesamte Steroid-Injektionen: {len(df_steroids)}')

In [ ]:
# Patient_ID
with pd.option_context('display.max_rows', None):
    print(df_clinical['Patient_ID'].sample(10))

In [ ]:
"""
for name, df in df_imp.items():
    buffer = io.StringIO()
    try:
        sys.stdout = buffer
        data_quality_report(df)
    finally:
        sys.stdout = sys.__stdout__
    
    with open(f'reports/{name}_report.txt', 'w', encoding='utf-8') as f:
        f.write(buffer.getvalue())

print("All reports saved")
"""

In [ ]:
#df_clinical['DAS28.18M'] = pd.to_numeric(df_clinical['DAS28.18M'])
df_clinical[['DAS28.0M', 'DAS28.3M', 'DAS28.6M', 'DAS28.9M', 'DAS28.12M', 'DAS28.18M'] ].dtypes.head(20)

In [ ]:
with pd.option_context('display.max_rows', None):
    print(df_clinical.dtypes)

In [ ]:
with pd.option_context('display.max_rows', None):
    print(df_clinical.columns)
    print(len(df_clinical.columns))

In [ ]:
print('Indexig booleans')
booleans = [
    'IM.STEROIDS.3MONTHS', 'ACPA.POSITIVE', 'RHUEMATOID.FACTOR',
    'Remission(<2.6DAS)', 'HighDisease(>4DAS)','ALCOHOL_Y_N',
    'CURENT SMOKER', 'ORAL.STEROIDS.3M'
]
print('Indexing objects')
object_to_numeric = [
    'DAS28.0M', 'DAS28.3M', 'DAS28.6M',
    'DAS28.9M', 'DAS28.12M', 'DAS28.18M',
    'Remission month', 'Symp_Duration', 'HAQ.0M', 'HAQ.6M', 'SDAI.0M', 'SDAI.6M', 'SDAI.12M',
    'BASOPHILS.0M', 'EOSINOPHILS.0M', 'HB.0M', 'LYMPHOCYTES.0M',
    'MONOCYTES.0M', 'NEUTROPHILS.0M', 'PLT.0M', 'WBC.0M', 'CRP.0M',
    'ESR.0M', 'FATIQUE.0M', 'PAIN.0M', 'TOTAL.SWOLLEN.0M',
    'TOTAL.TENDER.0M', 'BASOPHILS.6M', 'EOSINOPHILS.6M', 'HB.6M',
    'LYMPHOCYTES.6M', 'MONOCYTES.6M', 'NEUTROPHILS.6M', 'PLT.6M', 'WBC.6M',
    'CRP.6M', 'FATIQUE.6M', 'PAIN.6M', 'TOTAL.SWOLLEN.6M',
    'TOTAL.TENDER.6M', 'CRP.9M', 'TOTAL.SWOLLEN.9M', 'TOTAL.TENDER.9M',
    'HEIGHT', 'WEIGHT', 'AGE', 'InitialxRAYScore', 'FinalxRAYScore', 'Erosive',
    'Hep B serology wk 9 (IU/mL)'
]
print('indexing categories')
categorical = [
    'Region', 'Hub', 'REGION_HUB','Study','GENDER', 'RACE', 'vaccine centre'
]
print('indexing identifiers')
identifiers = ['Patient_ID', 'Digest']

print(f'{len(booleans) + len(object_to_numeric) + len(categorical) + len(identifiers)}')


In [ ]:
with pd.option_context('display.max_rows', None):
    print(df_clinical['ORAL.STEROIDS.3M'].value_counts())

In [ ]:
for col in booleans:
    df_clinical[col] = df_clinical[col].str.strip()
for col in booleans:
    df_clinical[col] = df_clinical[col].str.lower()

yes_val = ['yes', 'y', 'oral']
no_val = ['no', 'n']
na_val = ['nd', 'missing', 'unknown']
       
for col in booleans:
    df_clinical[col] = df_clinical[col].replace(
        {v: True for v in yes_val} |
        {v: False for v in no_val} |
        {v: pd.NA for v in na_val}
    ).astype('boolean')

for col in booleans:
    print(f'\n{col}:')
    print(df_clinical[col].value_counts(dropna=False))

In [ ]:
# parquet file extension to keep dtypes clean
df_clinical[booleans].to_parquet('cleaned_clinical.parquet', engine='fastparquet')

In [ ]:
# ! Store it
df_clinical.to_excel('cleaned_clinical.xlsx', index=False)
print('Gespeichert!')

In [ ]:
clean_clinical_sheet = pd.read_parquet('cleaned_clinical.parquet', engine='fastparquet')
clean_clinical_sheet.keys()
clean_clinical_sheet['ACPA.POSITIVE'].dtype

In [ ]:
for col in booleans:
    print(f'{col}: {df_clinical[col].dtype}')

In [ ]:
for col in object_to_numeric:
    non_numeric = pd.to_numeric(df_clinical[col], errors='coerce')
    mask = non_numeric.isna() & df_clinical[col].notna()
    problematic = df_clinical[col][mask].unique()
    if len(problematic) > 0:
        print(f'{col}: {problematic}')

# handling special cases
df_clinical['WEIGHT'] = df_clinical['WEIGHT'].replace('76/6',76.6)
df_clinical['Hep B serology wk 9 (IU/mL)'] = df_clinical['Hep B serology wk 9 (IU/mL)'].replace('>1000.0', 1000.0)
df_clinical['Remission month'] = df_clinical['Remission month'].replace('N', -1)

 # new column to check if remission was reached
df_clinical['reached_remission'] = df_clinical['Remission month'].apply(
    lambda x: pd.NA if pd.isna(x) else (False if x == -1 else True)
).astype('boolean')
 # at what month it was reached or empty if not reached
df_clinical['remission_at_month'] = df_clinical['Remission month'].replace(-1, pd.NA)

for col in object_to_numeric:
    df_clinical[col] = pd.to_numeric(df_clinical[col], errors='coerce')
# normalize HB value so everything is g/L
for col in ['HB.0M', 'HB.6M']:
    mask = df_clinical[col] < 30
    df_clinical.loc[mask, col] = df_clinical.loc[mask, col] * 10
print(df_clinical[object_to_numeric].dtypes.value_counts())
print(df_clinical.dtypes.value_counts())


In [ ]:
# parquet file extension to keep dtypes clean
df_clinical[object_to_numeric].to_parquet('cleaned_clinical.parquet', engine='fastparquet')

In [ ]:

for col in categorical:
    non_numeric = pd.to_numeric(df_clinical[col], errors='coerce')
    mask = non_numeric.isna() & df_clinical[col].notna()
    problematic = df_clinical[col][mask].unique()
    if len(problematic) > 0:
        print(f'{col}: {problematic}')



In [ ]:
df_clinical[identifiers].value_counts()

In [ ]:
"""
!!! Only run after running all others
Saving data in new excel sheet
"""
df_clinical.to_excel('../RA/cleaned_clinical.xlsx', index=False)